# Fractal Breakout

Fractal Breakout (Price Action) \
Detects S/R from **N-bar fractal pivots** — `indicators.detect_swing_highs` / `detect_swing_lows` (a strict `left`/`right` window), merged with `indicators.merge_price_levels`. \
This is the *indicators-layer* level source. It is **not** the dedicated `engine.level_detector` — that stateful horizontal-S/R detector (with invalidation tracking) powers the separate `level_breakout` strategy (see `level_breakout.ipynb`). \
Uses ATR(14) for the trailing stop. Best for scalping (1m-5m) and intraday (15m-1h).

__How the Fractal-Breakout Algorithm Determines Entry/Exit:__
- Detects swing highs (resistance) and lows (support) as N-bar fractal pivots with a configurable left/right window.
- Long Entry: Price closes above a significant resistance level + confirmation candle.
- Short Entry: Price closes below a significant support level + confirmation candle.
- Exit Long: Price closes below next support or ATR trailing stop hit.
- Exit Short: Price closes above next resistance or ATR trailing stop hit.
- Ideal for scalping/intraday when price respects liquidity levels.

## Configuration

In [ ]:
# --- path bootstrap: make 'engine' importable from any CWD ---
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "engine" / "__init__.py").exists():
    if root == root.parent: raise RuntimeError("project root not found")
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))

In [1]:
from engine.backtester import Backtester
from engine.trade_configurator import ACTIVE_TRADE
from engine.core import Signal, SignalAction
from engine.strategy_configurator import StrategyConfig
from engine.visualization import build_chart

In [ ]:
# Dataset is configured globally in engine/data_configurator.py (the ACTIVE spec).
# Edit ACTIVE there to change symbol / interval / window for every notebook at once.
from engine.data_configurator import ACTIVE, save_result

SYMBOL, INTERVAL = ACTIVE.symbol, ACTIVE.interval

In [ ]:
# Candles come from the shared cache via the global ACTIVE spec.
from engine.data_configurator import load_data

df = load_data()

## Fractal Breakout

It's a direction flip, not different entry logic:
- fractal_breakout has one entry signal: breakout (close crosses a fractal-pivot-derived S/R level).
- fractal_breakout and fractal_breakout_inv are two separate classes — ride the breakout vs fade it.
- The _inv is the same signal traded in the opposite direction.

In [ ]:
# Import fractal-breakout strategy
from engine.strategies import FractalBreakoutStrategy

In [ ]:
# Backtest fractal-breakout strategy
config = StrategyConfig()
strategy = FractalBreakoutStrategy(config)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=ACTIVE_TRADE)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Save metrics (JSON) + trade log (CSV) under data/results/<dataset signature>/
save_result(result, ACTIVE)

In [ ]:
# Fractal-breakout strategy chart
prepared = strategy.prepare(df)
signals = []
for t in result.trades:
    if t.entry_ts:
        signals.append(Signal(t.entry_ts, SignalAction.ENTRY, t.direction, t.entry_price))
    if t.exit_ts:
        signals.append(Signal(t.exit_ts, SignalAction.EXIT, t.direction, t.exit_price))

build_chart(
    prepared, signals,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()

## Inverse Fractal Breakout

In [ ]:
# Import inverse fractal-breakout strategy
from engine.strategies import InverseFractalBreakoutStrategy

In [ ]:
# Backtest inverse fractal-breakout strategy
config = StrategyConfig()
strategy = InverseFractalBreakoutStrategy(config)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=ACTIVE_TRADE)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Save metrics (JSON) + trade log (CSV) under data/results/<dataset signature>/
save_result(result, ACTIVE)

In [ ]:
# inverse fractal-breakout strategy chart
prepared = strategy.prepare(df)
signals = []
for t in result.trades:
    if t.entry_ts:
        signals.append(Signal(t.entry_ts, SignalAction.ENTRY, t.direction, t.entry_price))
    if t.exit_ts:
        signals.append(Signal(t.exit_ts, SignalAction.EXIT, t.direction, t.exit_price))

build_chart(
    prepared, signals,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()